# Notebook 03 — Validación de residuales del AR(1) y ajuste t-Student

Diagnóstico estadístico de los residuales $\varepsilon_t$ del segundo OLS (el AR(1) sobre los residuales estacionales del modelo Alaton).
Si los $\varepsilon_t$ rechazan normalidad, se ajusta una t-Student como alternativa robusta a colas pesadas para la simulación Monte Carlo del notebook 06.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

OUT_DIR = Path('../data/processed')

In [ ]:
# Cargar datos y parámetros calibrados en el notebook 02
df = pd.read_excel('../data/processed/Modelo_Completo.xlsx', parse_dates=['date'])
df = df.set_index('date')

cols = ['t_mean', 't_min', 't_max', 'precip']
df_Casanare = df[df['department'] == 'Casanare'][cols].copy()
df_Tolima   = df[df['department'] == 'Tolima'][cols].copy()

with open(OUT_DIR / 'params_alaton.json') as f:
    params_alaton = json.load(f)

# Decisión final del notebook 02 (basada en parsimonia + OOS):
#   Tolima   → 1 armónico (modelo base de Alaton)
#   Casanare → 2 armónicos (régimen bimodal por ZCIT)
N_ARMONICOS = {dpto: params_alaton[dpto]['n_armonicos'] for dpto in ['Tolima', 'Casanare']}
print("Modelo estacional elegido por departamento:")
for dpto, na in N_ARMONICOS.items():
    print(f"  {dpto:10s} → {na} armónico{'s' if na > 1 else ''}")

## Pruebas de validación

El proceso OU del modelo Alaton asume $\varepsilon_t \sim \mathcal{N}(0, \sigma^2)$ i.i.d.
Validamos sobre los **residuales del segundo OLS** (los que alimentan la simulación):

| Prueba | Hipótesis nula |
|---|---|
| Jarque-Bera | residuales normales (asimetría = 0, exceso de curtosis = 0) |
| Kolmogorov-Smirnov | residuales provienen de $\mathcal{N}(\hat{\mu}, \hat{\sigma}^2)$ |
| Ljung-Box (lags 10, 20) | no hay autocorrelación lineal en residuales |
| ARCH-LM (lags 5) | no hay heterocedasticidad condicional |

> **Nota sobre Shapiro-Wilk:** con n ≈ 5,800 observaciones diarias tiene poder estadístico tan alto que rechaza ante desviaciones triviales. Se usa Kolmogorov-Smirnov, que escala mejor a muestras grandes.

> **Comparación Normal vs t-Student:** se reportan log-likelihood, AIC y BIC. La Normal tiene k=2 parámetros (μ, σ), la t-Student tiene k=3 (df, loc, scale).

In [ ]:
def extraer_residuales(df_dpto, n_armonicos):
    """
    Corre el pipeline Alaton (componente estacional + AR(1)) y devuelve los
    residuales del segundo OLS ε_t. La especificación es consistente con el
    notebook 02: sin tendencia lineal (no significativa) y 1 o 2 armónicos
    según el departamento.
    """
    d = df_dpto.index.dayofyear

    regresores = {
        'sin': np.sin(2 * np.pi * d / 365),
        'cos': np.cos(2 * np.pi * d / 365),
    }
    if n_armonicos == 2:
        regresores['sin2'] = np.sin(4 * np.pi * d / 365)
        regresores['cos2'] = np.cos(4 * np.pi * d / 365)

    X_ols = sm.add_constant(pd.DataFrame(regresores, index=df_dpto.index))
    ols   = sm.OLS(df_dpto['t_mean'], X_ols).fit()
    X_t   = df_dpto['t_mean'] - ols.predict(X_ols)

    # AR(1) sobre los residuales estacionales
    lag = X_t.shift(1)
    ou_data = pd.concat([X_t, lag], axis=1).dropna()
    ou_data.columns = ['X', 'X_lag']
    ou = sm.OLS(ou_data['X'], sm.add_constant(ou_data['X_lag'])).fit()

    return pd.Series(ou.resid, name='residuales', index=ou_data.index)


# Asignaciones claras y consistentes con la decisión del notebook 02
resid_tolima   = extraer_residuales(df_Tolima,   n_armonicos=N_ARMONICOS['Tolima'])
resid_casanare = extraer_residuales(df_Casanare, n_armonicos=N_ARMONICOS['Casanare'])

print(f"Tolima   ({N_ARMONICOS['Tolima']} arm): n={len(resid_tolima):,}, "
      f"media={resid_tolima.mean():.4f}, std={resid_tolima.std():.4f}")
print(f"Casanare ({N_ARMONICOS['Casanare']} arm): n={len(resid_casanare):,}, "
      f"media={resid_casanare.mean():.4f}, std={resid_casanare.std():.4f}")

In [ ]:
def validar_residuales(resid, nombre_dpto, modelo_label):
    """
    Valida los residuales del AR(1) y compara ajustes Normal vs. t-Student
    mediante log-likelihood, AIC y BIC.
    """
    print(f"\n{'='*70}")
    print(f"  Validación de residuales ε_t | {nombre_dpto.upper()} | {modelo_label}")
    print(f"{'='*70}")

    n = len(resid)

    # ── Estadísticas descriptivas ────────────────────────────────────────────
    print(f"\n[0] Estadísticas descriptivas:")
    print(f"  n={n:,}  media={resid.mean():.4f}  std={resid.std():.4f}"
          f"  asimetría={stats.skew(resid):.3f}  curtosis exceso={stats.kurtosis(resid):.3f}")

    # ── Pruebas de normalidad ────────────────────────────────────────────────
    jb_stat, jb_p   = stats.jarque_bera(resid)
    ks_stat, ks_p   = stats.kstest(resid, 'norm', args=stats.norm.fit(resid))

    print(f"\n[1] Pruebas de normalidad:")
    print(f"  Jarque-Bera  : stat={jb_stat:.2f},  p={jb_p:.3e}")
    print(f"  KS (normal)  : stat={ks_stat:.4f}, p={ks_p:.3e}")

    rechaza_norm = (jb_p < 0.05) or (ks_p < 0.05)
    print(f"  → {'RECHAZA normalidad → se justifica t-Student' if rechaza_norm else 'No se rechaza normalidad'}")

    # ── Ljung-Box (autocorrelación lineal) ───────────────────────────────────
    lb = acorr_ljungbox(resid, lags=[10, 20], return_df=True)
    print(f"\n[2] Ljung-Box (autocorrelación lineal en ε_t):")
    for lag_val, row in lb.iterrows():
        print(f"  Lag {lag_val:2d}: Q={row['lb_stat']:.2f},  p={row['lb_pvalue']:.3e}")
    if (lb['lb_pvalue'] < 0.05).any():
        print("  → Se RECHAZA independencia: el AR(1) no elimina toda la autocorrelación lineal.")
        print("    Limitación: el modelo de simulación subestima la persistencia de anomalías.")
    else:
        print("  → No se rechaza independencia: residuales no autocorrelacionados.")

    # ── ARCH-LM (heterocedasticidad condicional) ─────────────────────────────
    arch_stat, arch_p, _, _ = het_arch(resid, nlags=5)
    print(f"\n[3] ARCH-LM (efectos ARCH en ε_t², lags=5):")
    print(f"  stat={arch_stat:.2f},  p={arch_p:.3e}")
    if arch_p < 0.05:
        print("  → Se RECHAZA: presencia de efectos ARCH (volatilidad condicional no constante).")
        print("    Limitación: σ constante subestima el riesgo en períodos de alta variabilidad.")
    else:
        print("  → No se rechaza: volatilidad homocedástica.")

    # ── Ajuste de distribuciones: Normal vs t-Student ────────────────────────
    mu_n, sig_n          = stats.norm.fit(resid)
    df_t, loc_t, scale_t = stats.t.fit(resid)

    ll_norm = stats.norm.logpdf(resid, mu_n, sig_n).sum()
    ll_t    = stats.t.logpdf(resid, df_t, loc_t, scale_t).sum()

    # AIC y BIC: penalizan complejidad. La Normal tiene k=2, la t k=3.
    k_norm, k_t = 2, 3
    aic_norm = 2 * k_norm - 2 * ll_norm;  bic_norm = k_norm * np.log(n) - 2 * ll_norm
    aic_t    = 2 * k_t    - 2 * ll_t;     bic_t    = k_t    * np.log(n) - 2 * ll_t

    print(f"\n[4] Ajuste de distribuciones (selección por AIC/BIC):")
    print(f"  Normal   : μ={mu_n:.4f},   σ={sig_n:.4f}"
          f"   logL={ll_norm:>10.1f}   AIC={aic_norm:>10.1f}   BIC={bic_norm:>10.1f}")
    print(f"  t-Student: df={df_t:.2f},  loc={loc_t:.4f},  scale={scale_t:.4f}"
          f"   logL={ll_t:>10.1f}   AIC={aic_t:>10.1f}   BIC={bic_t:>10.1f}")
    print(f"  ΔAIC (t − Normal) = {aic_t - aic_norm:+.1f}    ΔBIC = {bic_t - bic_norm:+.1f}")
    mejor = 't-Student' if aic_t < aic_norm else 'Normal'
    print(f"  → Selección AIC: {mejor}")

    if df_t < 2:
        print(f"  AVISO: df={df_t:.2f} < 2 → la t ajustada tiene varianza teórica infinita.")
        print("    Para simulación, usar df=max(df_t, 2+ε) o truncar colas.")
    elif df_t < 5:
        print(f"  Colas muy pesadas (df={df_t:.2f}): la Normal subestima eventos extremos.")

    # ── Visualización ────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f'Diagnóstico de ε_t — {nombre_dpto} ({modelo_label})',
                 fontsize=13, fontweight='bold')

    ax_hist = plt.subplot2grid((2, 2), (0, 0), rowspan=2)
    ax_qq   = plt.subplot2grid((2, 2), (0, 1))
    ax_acf  = plt.subplot2grid((2, 2), (1, 1))

    ax_hist.hist(resid, bins=60, density=True, alpha=0.35, color='slategrey', label='ε_t observado')
    x_grid = np.linspace(resid.quantile(0.001), resid.quantile(0.999), 500)
    ax_hist.plot(x_grid, stats.norm.pdf(x_grid, mu_n, sig_n),
                 color='red', lw=2, label=f'Normal (σ={sig_n:.2f})')
    ax_hist.plot(x_grid, stats.t.pdf(x_grid, df_t, loc_t, scale_t),
                 color='steelblue', lw=2, ls='--', label=f't-Student (df={df_t:.1f})')
    ax_hist.set_title('Distribución empírica vs. ajustes', fontsize=11)
    ax_hist.set_xlabel('ε_t (°C)'); ax_hist.set_ylabel('Densidad')
    ax_hist.legend(fontsize=9); ax_hist.grid(alpha=0.2)

    sm.qqplot(resid.values, stats.norm, fit=True, line='45', ax=ax_qq,
              markerfacecolor='slategrey', markeredgecolor='none', alpha=0.4, markersize=2)
    ax_qq.get_lines()[-1].set_color('red')
    ax_qq.set_box_aspect(1)
    ax_qq.set_title('Q-Q plot vs. Normal', fontsize=11)
    ax_qq.set_xlabel('Cuantiles teóricos'); ax_qq.set_ylabel('Cuantiles empíricos')
    ax_qq.grid(alpha=0.2)

    max_lags = min(30, n // 4)
    plot_acf(resid, ax=ax_acf, lags=max_lags, color='steelblue',
             vlines_kwargs={'colors': 'steelblue', 'linewidth': 1.2})
    ax_acf.set_title(f'ACF de ε_t (lags 1–{max_lags})', fontsize=11)
    ax_acf.set_xlabel('Lag'); ax_acf.set_ylabel('Autocorrelación')
    ax_acf.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

    return {
        'jb_p': jb_p, 'ks_p': ks_p,
        'lb_p_10': lb['lb_pvalue'].iloc[0], 'lb_p_20': lb['lb_pvalue'].iloc[1],
        'arch_p': arch_p,
        'rechaza_normalidad': rechaza_norm,
        'normal_params': {'mu': mu_n, 'sigma': sig_n},
        't_params':      {'df': df_t, 'loc': loc_t, 'scale': scale_t},
        'll_norm': ll_norm, 'll_t': ll_t,
        'aic_norm': aic_norm, 'aic_t': aic_t,
        'bic_norm': bic_norm, 'bic_t': bic_t,
        'mejor_distribucion': mejor,
    }

In [ ]:
res_tolima   = validar_residuales(resid_tolima,   'Tolima',   f"Alaton {N_ARMONICOS['Tolima']} armónico")
res_casanare = validar_residuales(resid_casanare, 'Casanare', f"Alaton {N_ARMONICOS['Casanare']} armónicos")

## Exportación de parámetros de las innovaciones

Para cada departamento se exportan tanto los parámetros Normal como t-Student.
El notebook 06 elegirá según el criterio AIC reportado arriba (con respaldo manual si la t tiene df < 2).

In [ ]:
params_innovaciones = {
    'Tolima': {
        'normal':   res_tolima['normal_params'],
        't':        res_tolima['t_params'],
        'mejor':    res_tolima['mejor_distribucion'],
        'aic_norm': res_tolima['aic_norm'],  'aic_t': res_tolima['aic_t'],
    },
    'Casanare': {
        'normal':   res_casanare['normal_params'],
        't':        res_casanare['t_params'],
        'mejor':    res_casanare['mejor_distribucion'],
        'aic_norm': res_casanare['aic_norm'],  'aic_t': res_casanare['aic_t'],
    },
}

out_path = OUT_DIR / 'params_innovaciones.json'
with open(out_path, 'w') as f:
    json.dump(params_innovaciones, f, indent=2, ensure_ascii=False)

print(f"Parámetros de innovaciones exportados a: {out_path}")
print(json.dumps(params_innovaciones, indent=2, ensure_ascii=False))